In [17]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pandas as pd
import os

In [18]:
file_path = "data/IMDB_Dataset.csv"
df = pd.read_csv(file_path)
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [19]:
import re
from bs4 import BeautifulSoup

stop_words = ['ourselves','hers','between','yourself','but','again','there','about','once','during',
              'out','very','having','with','they','own','an','be','some','for','do','its','yours',
              'such','into','of','most','itself','other','off','is','s','am','or','who','as','from',
              'him','each','the','themselves','until','below','are','we','these','your','his',
              'through','me','were','her','more','himself','this','down','should','our','their',
              'while','above','both','up','to','ours','had','she','all','when','at','any','before',
              'them','same','and','been','have','in','will','on','does','yourselves','then','that',
              'because','what','over','why','so','can','did','now','under','he','you','herself',
              'has','just','where','too','only','myself','which','those','i','after','few','whom',
              't','being','if','theirs','my','against','a','by','doing','it','how','further','was',
              'here','than']

def clean(df):
    def preprocess(text):
        text = BeautifulSoup(text, "html.parser").get_text()
        text = text.lower()
        text = text.replace("'", "")
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)
        tokens = text.split()
        tokens = [w for w in tokens if w not in stop_words]
        tokens = [w for w in tokens if not re.fullmatch(r'([a-z])\1+', w)]
        text = " ".join(tokens)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    df["review"] = df["review"].astype(str).apply(preprocess)
    return df

df = clean(df)
df.head()

,review,sentiment
0,one reviewers mentioned watching oz episode yo...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [20]:
labels = df.sentiment.tolist()
documents = df.review.tolist()
print(documents[:5])
print(labels[:5])

['one reviewers mentioned watching oz episode youll hooked right exactly happened first thing struck oz brutality unflinching scenes violence set right word go trust not show faint hearted timid show pulls no punches regards drugs sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focuses mainly emerald city experimental section prison cells glass fronts face inwards privacy not high agenda em city home many aryans muslims gangstas latinos christians italians irish scuffles death stares dodgy dealings shady agreements never far away would say main appeal show due fact goes shows wouldnt dare forget pretty pictures painted mainstream audiences forget charm forget romance oz doesnt mess around first episode ever saw struck nasty surreal couldnt say ready watched developed taste oz got accustomed high levels graphic violence not violence injustice crooked guards wholl sold nickel inmates wholl kill order get away well mannered middle 

In [21]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    documents, labels, test_size=0.3, random_state=42
)

In [22]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)


In [23]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 3499543 stored elements and shape (35000, 92497)>
  Coords	Values
  (0, 54405)	1
  (0, 48292)	1
  (0, 83625)	1
  (0, 17561)	1
  (0, 78121)	1
  (0, 54262)	1
  (0, 63444)	1
  (0, 57884)	1
  (0, 17558)	1
  (0, 77670)	1
  (0, 47849)	1
  (0, 23771)	1
  (0, 3895)	1
  (0, 13836)	1
  (0, 90619)	1
  (0, 37575)	1
  (0, 3064)	2
  (0, 83613)	1
  (0, 2086)	1
  (0, 89246)	1
  (0, 68607)	1
  (0, 39720)	1
  (0, 62193)	1
  (0, 47431)	1
  (0, 37531)	1
  :	:
  (34999, 26925)	1
  (34999, 28124)	1
  (34999, 37810)	1
  (34999, 84955)	1
  (34999, 49342)	1
  (34999, 19920)	1
  (34999, 78280)	1
  (34999, 21296)	1
  (34999, 80540)	1
  (34999, 27435)	1
  (34999, 25616)	2
  (34999, 70893)	1
  (34999, 17046)	1
  (34999, 30716)	1
  (34999, 63667)	1
  (34999, 13087)	1
  (34999, 81466)	1
  (34999, 83572)	1
  (34999, 34514)	1
  (34999, 50719)	1
  (34999, 26928)	1
  (34999, 21779)	1
  (34999, 13565)	1
  (34999, 59186)	1
  (34999, 15896)	1


In [24]:
model = MultinomialNB()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
probs = model.predict_proba(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.8584


In [25]:
print(predictions[:5])
print(probs[:5])

['positive' 'positive' 'negative' 'positive' 'negative']
[[3.20695997e-01 6.79304003e-01]
 [4.44335107e-10 1.00000000e+00]
 [9.99999999e-01 1.06116482e-09]
 [3.60202085e-07 9.99999640e-01]
 [9.99962753e-01 3.72472303e-05]]


In [26]:
results_df = pd.DataFrame({
    "review": X_test_raw,   
    "true_label": y_test,
    "predicted_label": predictions,
    "prob_negative": probs[:, 0],
    "prob_positive": probs[:, 1],
})

results_df["prob_negative"] = results_df["prob_negative"].round(5)
results_df["prob_positive"] = results_df["prob_positive"].round(5)

results_df

,review,true_label,predicted_label,prob_negative,prob_positive
0,really liked summerslam due look arena curtain...,positive,positive,0.32070,0.67930
1,not many television shows appeal quite many di...,positive,positive,0.00000,1.00000
2,film quickly gets major chase scene ever incre...,negative,negative,1.00000,0.00000
3,jane austen would definitely approve one gwyne...,positive,positive,0.00000,1.00000
4,expectations somewhat high went see movie thou...,negative,negative,0.99996,0.00004
...,...,...,...,...,...
14995,landscape battle opens escaping prisoners snow...,positive,positive,0.00000,1.00000
14996,jake speed amusing parody indiana jones advent...,positive,positive,0.00020,0.99980
14997,plan b appearance quickly made unedited sloppy...,negative,negative,0.99912,0.00088
14998,one perks job things slow watch movie downstai...,positive,positive,0.00036,0.99964


In [27]:
sample_df = results_df[
    (results_df["prob_negative"] > 0) &
    (results_df["prob_negative"] < 1) &
    (results_df["prob_positive"] > 0) &
    (results_df["prob_positive"] < 1)
]
sample_df.head(50)

,review,true_label,predicted_label,prob_negative,prob_positive
0,really liked summerslam due look arena curtain...,positive,positive,0.32070,0.67930
4,expectations somewhat high went see movie thou...,negative,negative,0.99996,0.00004
10,jeez immensely boring leading man christian sc...,negative,negative,0.99999,0.00001
13,movie stinks majorly reason gave graphics semi...,negative,negative,0.99999,0.00001
15,movie starts somewhat slowly gets running towa...,positive,positive,0.00110,0.99890
16,slightly uneven entry one standout sequence in...,positive,positive,0.45988,0.54012
17,first introduced john waters films seeing fema...,positive,positive,0.04244,0.95756
18,movie good acting virtually cast gripping stor...,positive,positive,0.35305,0.64695
19,cant help notice negative reviews movie gotten...,positive,negative,0.81610,0.18390
20,production quality cast premise authentic new ...,positive,positive,0.00042,0.99958


In [28]:
print(sample_df.review.loc[14997])
sample_df.loc[14997]

plan b appearance quickly made unedited sloppy script movie attempt outing actors involved outing nixed start another mafia based comedy nothing new lowers standard participating joe maloni paul sorvino crime boss concerned clothing appearances business control personal assistant mario anthony desando dumber dirt ignorance supposed funny maloni whacked one debtors happens married bookish fran diane keaton maloni takes fran assistant work dead husbands debt malonis hit man fran afraid shadow unable carry malonis assignments electing instead transport whackees florida hide brother james house figure next alternative killing three candidates called plan b plan kill ending wholly predictable every line assigned script characters diane keaton made lot fine films one talented actresses comediennes screams rants twitches way ridiculous part quickly becomes annoying watch paul sorvino well paul sorvino type cast mobster films supporting cast likewise allowed play balcony broadest slapstick pra

review             plan b appearance quickly made unedited sloppy...
true_label                                                  negative
predicted_label                                             negative
prob_negative                                                0.99912
prob_positive                                                0.00088
Name: 14997, dtype: object

In [29]:
print(sample_df.review.loc[98])
sample_df.loc[98]

ok even cant stand liza movie truly hilarious scenes john gielgud make liza one true romantic comedy classics th century dudley moore makes drunk irresponsible look cute amusing damn fun watch one liners best


review             ok even cant stand liza movie truly hilarious ...
true_label                                                  positive
predicted_label                                             positive
prob_negative                                                0.00441
prob_positive                                                0.99559
Name: 98, dtype: object

In [30]:
new_review = ["this movie was good and i enjoyed watching it"]
v = vectorizer.transform(new_review)
new_review_pred_label = model.predict(v)[0]
new_review_probs = model.predict_proba(v)[0]
print(f"Review: {new_review[0]}")
print(f"Predicted label: {new_review_pred_label}")
print(f"neg: {new_review_probs[0]:.4f}, pos: {new_review_probs[1]:.4f}")


Review: this movie was good and i enjoyed watching it
Predicted label: positive
neg: 0.4452, pos: 0.5548


In [54]:
import pandas as pd
import numpy as np

feature_names = vectorizer.get_feature_names_out()
class_order = model.classes_ 

neg_idx = np.where(class_order == "negative")[0][0]
pos_idx = np.where(class_order == "positive")[0][0]

log_prob_neg = model.feature_log_prob_[neg_idx]
log_prob_pos = model.feature_log_prob_[pos_idx]

weights_df = pd.DataFrame({
    "token": feature_names,
    "log_prob_negative": log_prob_neg,
    "log_prob_positive": log_prob_pos
})

weights_df["sentiment_score"] = weights_df["log_prob_positive"] - weights_df["log_prob_negative"]
words = [
 "amazing", "enjoy",  "beautiful", "terrific", "fantastic"
]

subset = weights_df[weights_df["token"].isin(words)]
print(subset)
print("\n")
negative_words = [
    "bad", "terrible", "awful", "boring", "worst",
]
subset = weights_df[weights_df["token"].isin(negative_words)]
print(subset)

           token  log_prob_negative  log_prob_positive  sentiment_score
2294     amazing          -8.714109          -7.419604         1.294504
6715   beautiful          -7.898306          -6.976546         0.921760
25964      enjoy          -7.743288          -7.260562         0.482727
28186  fantastic          -9.293143          -7.894996         1.398147
81571   terrific          -9.938434          -8.458822         1.479612


          token  log_prob_negative  log_prob_positive  sentiment_score
5196      awful          -6.939436          -9.254053        -2.314617
5489        bad          -5.365555          -6.733196        -1.367641
9304     boring          -6.951620          -8.508562        -1.556942
81564  terrible          -7.005373          -8.906296        -1.900924
91036     worst          -6.452746          -8.864287        -2.411541
